# Problem Set 2
## Problem 2: Loopy Belief Propagation
### 1. Sum Product Algorithm

In [ ]:
from collections import defaultdict
from copy import deepcopy
import math
import numpy as np

def sumprod(M, s, t, w, its):
    """Sum-product algorithm

    Arguments:
        M {n x n matrix} -- adjacency matrix of graph G
        s {integer} -- vertex in set A
        t {integer} -- vertex in set B
        w {n x n matrix} -- weights for each edge
        its {integer} -- number of iterations

    Returns:
        The approximate partition function attained after `its` iterations by
        plugging the appropriately normalized beliefs into the Bethe free energy
    """

    n = M.shape[0]
    edges = []
    for i in range(n):
        for j in range(n):
            if M[i,j] == 1:
                edges.append((i,j))

    phi = lambda i, x_i: x_i if i == s else (1 - x_i if i == t else 1)
    psi = lambda i, j, x_i, x_j: w[i, j] ** int(x_i != x_j)

    # n x n matrix
    # Stores messages from factor nodes (C) to vertices (i).
    # C_i_messages[C, i] represents the message from the factor C
    # to the vertex i
    C_i_messages = defaultdict(lambda: 1)
    new_C_i_messages = defaultdict(lambda: 1)

    # n x n matrix
    # Stores messages from vertices (i) to factor nodes (C).
    # i_C_messages[i, C] represents the message from the vertex i
    # to the factor C
    i_C_messages = defaultdict(lambda: 1)

    for it in range(its):
        # Messages from C to i
        for (C, i) in edges:
            for x_i in range(2):
                C_i_sum = 0

                # Sum over assignments to x_C
                for x_C in range(2):
                    # Product over k in C \ i = {C}
                    # Would be a loop if I was using cliques and not just edges
                    C_i_sum += psi(C, i, x_C, x_i) * i_C_messages[C, (C, i), x_C]

                new_C_i_messages[(C, i), i, x_i] = C_i_sum

        # Messages from i to C
        for (i, C) in edges:
            for x_i in range(2):
                prod = phi(i, x_i)

                # Product over messages into i
                for (C_prime, k) in edges:
                    # Exclude C -> i
                    if k == i and C_prime != C:
                        prod *= C_i_messages[(C_prime, k), k, x_i]

                i_C_messages[i, (i, C), x_i] = prod

        C_i_messages = deepcopy(new_C_i_messages)

        # Scale
        max_val = max(map(abs, [*C_i_messages.values(), *i_C_messages.values()]))
        if max_val > 0:
            for i in C_i_messages:
                C_i_messages[i] = max(1e-50, C_i_messages[i] / max_val)
            for i in i_C_messages:
                i_C_messages[i] = max(1e-50, i_C_messages[i] / max_val)


        # print()
        # print(f"{it} i -> C")
        # print("-----------------------------------")
        # for i, (i, C), x_i in i_C_messages:
        #     # Concatenate the f-string with the joined string.
        #     print(f"({i}, ({i}, {C})) {x_i}: | {round(i_C_messages[i, (i, C), x_i], 4)}")
        # print()
        # print(f"{it} C -> i")
        # print("-----------------------------------")
        # for (i, j), j, x_i in C_i_messages:
        #     # Concatenate the f-string with the joined string.
        #     print(f"(({i}, {j}), {j}) {x_i}: | {round(C_i_messages[(i, j), j, x_i], 4)}")
        # print()




    # Calculate beliefs for all i
    beliefs = np.empty(n)
    for i in range(n):
        belief0 = phi(i, 0)
        belief1 = phi(i, 1)

        for (C, k) in edges:
            if k == i:
                belief0 *= C_i_messages[(C, k), k, 0]
                belief1 *= C_i_messages[(C, k), k, 1]
        beliefs[i] = belief1 / (belief0 + belief1)

    # Calculate beliefs for all C
    C_beliefs = defaultdict(int)
    for (i, j) in edges:
        if (j, i) in C_beliefs:
            continue

        total = 0

        for x_i in range(2):
            for x_j in range(2):
                prob = psi(i, j, x_i, x_j)
                # Message i -> C (x_i)
                prob *= i_C_messages[i, (i, j), x_i]
                # Message j -> C (x_j)
                prob *= i_C_messages[j, (j, i), x_j]

                C_beliefs[(i, j), (x_i, x_j)] = prob
                total += prob

        # Normalize
        for x_i in range(2):
            for x_j in range(2):
                C_beliefs[(i, j), (x_i, x_j)] /= total

    bethe = 0
    eps = 1e-7

    for i in range(n):
        belief0 = 1 - beliefs[i]
        belief1 = beliefs[i]

        if belief0 > eps:
            bethe -= belief0 * math.log(belief0)

        if belief1 > eps:
            bethe -= belief1 * math.log(belief1)

    for (i, j) in edges:
        if i < j:
            for x_i in range(2):
                for x_j in range(2):
                    belief_i = beliefs[i]
                    belief_j = beliefs[j]
                    C_belief = C_beliefs[(i, j), (x_i, x_j)]

                    if x_i == 0:
                        belief_i = 1 - belief_i
                    if x_j == 0:
                        belief_j = 1 - belief_j

                    if C_belief > eps and belief_i * belief_j > 0:
                        bethe -= C_belief * math.log(C_belief / (belief_i * belief_j))

                    bethe += C_belief * math.log(psi(i, j, x_i, x_j))

    return beliefs, round(math.exp(bethe), 3)

In [136]:
M = np.array([
    [0,1,0,0,1],
    [1,0,1,0,0],
    [0,1,0,1,0],
    [0,0,1,0,1],
    [1,0,0,1,0]
], dtype = np.float64)

# Weights (w):
w = np.array([
    [0,2,0,0,4],
    [2,0,3,0,0],
    [0,3,0,5,0],
    [0,0,5,0,6],
    [4,0,0,6,0]
], dtype = np.float64)

s = 0
t = 4
its = 100

beliefs, partition_function = sumprod(M, s, t, w, its)
print(f"Approximate Partition Function: {partition_function}")
print(f"Beliefs: {beliefs}")

Approximate Partition Function: 928.0
Beliefs: [1.         0.44827586 0.33189655 0.82758621 0.        ]


In [150]:
import math

M = np.array([[0, 1, 0],  # a
              [1, 0, 1],  # b
              [0, 1, 0]], # c
              dtype = np.float64)

w = np.array([[0, math.exp(1), 0],  # a
              [math.exp(1), 0, math.exp(1)],  # b
              [0, math.exp(1), 0]], # c
              dtype = np.float64)

# w = np.array([[0, 0.5, 0],  # a
#               [0.5, 0, .75],  # b
#               [0, .75, 0]], # c
#               dtype = np.float64)

s = 0
t = 2
its = 100

beliefs, partition_function = sumprod(M, s, t, w, its)
print(f"Approximate Partition Function: {partition_function}")
print(f"Beliefs: {beliefs}")

Approximate Partition Function: 5.437
Beliefs: [1.  0.5 0. ]


In [138]:
M = np.array([[0, 1, 0],  # a
              [1, 0, 1],  # b
              [0, 1, 0]], # c
              dtype = np.float64)

w = np.array([[0, 5, 0],  # a
              [5, 0, 3],  # b
              [0, 3, 0]], # c
              dtype = np.float64)

s = 0
t = 2
its = 100

beliefs, partition_function = sumprod(M, s, t, w, its)
print(f"Approximate Partition Function: {partition_function}")
print(f"Beliefs: {beliefs}")

Approximate Partition Function: 8.0
Beliefs: [1.    0.375 0.   ]


In [155]:
#              a, b, c, d
M = np.array([[0, 1, 1, 0],  # a
              [1, 0, 0, 1],  # b
              [1, 0, 0, 1],  # c
              [0, 1, 1, 0]], # b
              dtype = np.float64)

#              a, b, c, d
w = np.array([[0, 1, 2, 0],  # a
              [1, 0, 0, 1],  # b
              [2, 0, 0, 1],  # c
              [0, 1, 1, 0]], # d
              dtype = np.float64)

s = 0
t = 3
its = 100

beliefs, partition_function = sumprod(M, s, t, w, its)
print(f"Approximate Partition Function: {partition_function}")
print(f"Beliefs: {beliefs}")

Approximate Partition Function: 6.0
Beliefs: [1.         0.5        0.33333333 0.        ]


In [140]:
#              a, b, c
M = np.array([[0, 1, 0],  # a
              [1, 0, 1],  # b
              [0, 1, 0]], # c
              dtype = np.float64)

#              a, b, c
w = np.array([[0, .2, 0],  # a
              [.2, 0, .9],  # b
              [0, .9, 0]], # c
              dtype = np.float64)

s = 0
t = 2
its = 100

beliefs, partition_function = sumprod(M, s, t, w, its)
print(f"Approximate Partition Function: {partition_function}")
print(f"Beliefs: {beliefs}")

Approximate Partition Function: 1.1
Beliefs: [1.         0.81818182 0.        ]


### 2. Max Product Algorithm

In [ ]:
import numpy as np

def maxprod(M, s, t, w, its):
    """Max-product algorithm

    Arguments:
        M {n x n matrix} -- adjacency matrix of graph G
        s {integer} -- vertex in set A
        t {integer} -- vertex in set B
        w {n x n matrix} -- weights for each edge
        its {integer} -- number of iterations

    Returns:
        The approximate maximum cut attained after `its` iterations by
        returning the assignment that maximizes each singleton belief
    """

    n = M.shape[0]
    edges = []
    for i in range(n):
        for j in range(n):
            if M[i,j] == 1:
                edges.append((i,j))

    phi = lambda i: 1 if i == s else (0 if i == t else 0.5)
    psi = lambda i, j, x_i, x_j: w[i, j] ** int(x_i != x_j)

    # n x n matrix
    # Stores messages from factor nodes (C) to vertices (i).
    # C_i_messages[C, i] represents the message from the factor C
    # to the vertex i
    C_i_messages = M.copy()

    # n x n matrix
    # Stores messages from vertices (i) to factor nodes (C).
    # i_C_messages[i, C] represents the message from the vertex i
    # to the factor C
    i_C_messages = M.copy()

    for it in range(its):
        new_C_i_messages = C_i_messages.copy()

        # Messages from C to i
        for (C, i) in edges:
            max0 = 0
            max1 = 0

            # Sum over assignments to x_C
            for x_C in range(2):
                # Product over k in C \ i = {C}
                # Would be a loop if I was using cliques and not just edges
                k_C_message = i_C_messages[C, i]
                if x_C == 0:
                    k_C_message = (1 - k_C_message)

                max0 = max(max0, psi(C, i, x_C, 0) * k_C_message)
                max1 = max(max1, psi(C, i, x_C, 1) * k_C_message)

            # Normalize
            new_C_i_messages[C, i] = max1 / (max0 + max1)

        # Messages from i to C
        for (i, C) in edges:
            prod0 = 1 - phi(i)
            prod1 = phi(i)

            # Product over messages into i
            for (C_prime, k) in edges:
                # Exclude C -> i
                if k == i and C_prime != C:
                    prod0 *= 1 - C_i_messages[C_prime, k]
                    prod1 *= C_i_messages[C_prime, k]

            # Normalize
            prod0 += 1e-12
            prod1 += 1e-12
            i_C_messages[i, C] = prod1 / (prod0 + prod1)

        C_i_messages = new_C_i_messages.copy()

    beliefs = np.empty(n)
    # Calculate beliefs for all x_i
    for i in range(n):
        belief0 = 1 - phi(i)
        belief1 = phi(i)

        for (C, k) in edges:
            if k == i:
                belief0 *= 1 - C_i_messages[C, k]
                belief1 *= C_i_messages[C, k]
        beliefs[i] = 0 if belief0 + belief1 == 0 else belief1 / (belief0 + belief1)

    print(beliefs)

    # Convert beliefs into assignments
    return [(0.5 if b == 0.5 else int(b > 0.5)) for b in beliefs]

In [44]:
# Adjacency for a "star": 0 connected to 1,2,3
n = 4
M_test = np.zeros((n,n), dtype = np.float64)
for (i,j) in [(0,1), (0,2), (0,3)]:
    M_test[i,j] = 1
    M_test[j,i] = 1

# Let's do w=2 for edges (0,1) and (0,2), and w=3 for (0,3)
w_test = np.zeros((n,n), dtype = np.float64)
edges = [(0,1,2.0), (0,2,2.0), (0,3,3.0)]
for (i,j,wval) in edges:
    w_test[i,j] = wval
    w_test[j,i] = wval

s_idx = 0
t_idx = 3

print(maxprod(M_test, s_idx, t_idx, w_test, its=6))

[1.         0.33333333 0.33333333 0.        ]
[1, 0, 0, 0]


In [27]:
#              a, b, c, d
M = np.array([[0, 1, 1, 0],  # a
              [1, 0, 0, 1],  # b
              [1, 0, 0, 1],  # c
              [0, 1, 1, 0]], # b
              dtype = np.float64)

#              a, b, c, d
w = np.array([[0, 2, 1, 0],  # a
              [2, 0, 0, 1],  # b
              [1, 0, 0, 1],  # c
              [0, 1, 1, 0]], # d
              dtype = np.float64)

s = 0
t = 3
its = 100

print("Approximate Maximum Cut:")
print(maxprod(M, s, t, w, its))

Approximate Maximum Cut:
[1.         0.33333333 0.5        0.        ]
[1, 0, 0.5, 0]


In [42]:
M = np.array([[0, 1, 0],  # a
              [1, 0, 1],  # b
              [0, 1, 0]], # c
              dtype = np.float64)

w = np.array([[0, 5, 0],  # a
              [5, 0, 3],  # b
              [0, 3, 0]], # c
              dtype = np.float64)

s = 0
t = 2
its = 100

print("Approximate Maximum Cut:")
print(maxprod(M, s, t, w, its))

Approximate Maximum Cut:
[1.    0.375 0.   ]
[1, 0, 0]


In [30]:
#              a, b, c
M = np.array([[0, 1, 0],  # a
              [1, 0, 1],  # b
              [0, 1, 0]], # c
              dtype = np.float64)

#              a, b, c
w = np.array([[0, .2, 0],  # a
              [.2, 0, .9],  # b
              [0, .9, 0]], # c
              dtype = np.float64)

s = 0
t = 2
its = 100

print("Approximate Maximum Cut:")
print(maxprod(M, s, t, w, its))

Approximate Maximum Cut:
[1.         0.81818182 0.        ]
[1, 1, 0]


In [31]:
M = np.array([
    [0,1,0,0,1],
    [1,0,1,0,0],
    [0,1,0,1,0],
    [0,0,1,0,1],
    [1,0,0,1,0]
], dtype = np.float64)

# Weights (w):
w = np.array([
    [0,2,0,0,4],
    [2,0,3,0,0],
    [0,3,0,5,0],
    [0,0,5,0,6],
    [4,0,0,6,0]
], dtype = np.float64)

s = 0
t = 4
its = 100

print("Approximate Maximum Cut:")
print(maxprod(M, s, t, w, its))

Approximate Maximum Cut:
[1.         0.6        0.28571429 0.75       0.        ]
[1, 1, 0, 1, 0]


In [32]:
# Case 10
M = np.array([
    [0,1,1],
    [1,0,1],
    [1,1,0]
], dtype = np.float64)

w = np.array([
    [0,2,4],
    [2,0,3],
    [4,3,0]
], dtype = np.float64)

s = 0
t = 2
its = 100

print("Approximate Maximum Cut:")
print(maxprod(M, s, t, w, its))

Approximate Maximum Cut:
[1.  0.6 0. ]
[1, 1, 0]


In [36]:
# Case 9
M = np.array([
    [0,1,0,0,0],
    [1,0,1,0,0],
    [0,1,0,1,0],
    [0,0,1,0,1],
    [0,0,0,1,0]
], dtype = np.float64)

w = np.array([
    [0,2,0,0,0],
    [2,0,3,0,0],
    [0,3,0,4,0],
    [0,0,4,0,5],
    [0,0,0,5,0]
], dtype = np.float64)

s = 0
t = 4
its = 100

print("Approximate Maximum Cut:")
print(maxprod(M, s, t, w, its))

Approximate Maximum Cut:
[1.         0.6        0.33333333 0.71428571 0.        ]
[1, 1, 0, 1, 0]
